# 01 The mean model (our baseline)

**What it is:** for every variable (vcd, tcd, substrate, byproduct) and every day, take the *average of the training batches*. That curve is the prediction for **every** batch. The model never looks at the batch itself, only at the time.

**Why it matters:** it is the bar every other model must beat. Ridge, gbm and blend all start from this curve and only learn a correction on top.

We use the **geometric mean** (average of the logs, converted back), because the leaderboard score is a % error (MAPE).

We score it with the team's frozen 5-fold split, on the `dev` batches only. The holdout stays closed.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TARGETS = ["vcd", "tcd", "substrate", "byproduct"]

df = pd.read_csv(ROOT / "data" / "bioreaction_remaining.csv")
splits = pd.read_csv(ROOT / "outputs" / "splits.csv")

# measured rows after day 0 of the dev batches (same 507 rows as the shared table)
dev = df[df.vcd.notna() & (df.elapsed_time > 0)].merge(splits[["batch_id", "role", "fold"]], on="batch_id")
dev = dev[dev.role == "dev"].reset_index(drop=True)
dev["day"] = dev.elapsed_time.round().astype(int)
assert len(dev) == 507 and dev.batch_id.nunique() == 39
print(len(dev), "rows,", dev.batch_id.nunique(), "dev batches, folds:", sorted(dev.fold.unique()))

## 1. The model and the score

In [ ]:
def mean_curve(train, t):
    # geometric mean per day (mean of logs), then straight lines between days
    m = np.exp(np.log(train[TARGETS]).groupby(train.day).mean())
    return np.column_stack([np.interp(t, m.index.values, m[c].values) for c in TARGETS])


def mape(obs, pred):
    # leaderboard metric: % error per variable, then the average of the 4 (lower is better)
    s = 100 * np.mean(np.abs((obs - pred) / obs), axis=0)
    return dict(zip(TARGETS, s)) | {"mean": s.mean()}

## 2. Score with the frozen 5-fold CV
Each fold is predicted by a model that never saw those batches. We keep every prediction (out-of-fold) so we can look at the errors in detail.

In [ ]:
oof = dev.copy()
rows = []
for k in sorted(dev.fold.unique()):
    tr, va = dev[dev.fold != k], dev[dev.fold == k]
    p = mean_curve(tr, va.elapsed_time.values)
    rows.append(dict(fold=k, **mape(va[TARGETS].values, p)))
    for j, c in enumerate(TARGETS):
        oof.loc[va.index, f"pred_{c}"] = p[:, j]

res = pd.DataFrame(rows)
print("MAPE (%) of the mean curve, over the 5 folds")
pd.DataFrame({"mean": res[TARGETS + ["mean"]].mean(), "std": res[TARGETS + ["mean"]].std()}).round(2).T

## 3. Interpretation of the score

*(numbers from the run at the time of writing; mean over the 5 folds, +/- is the std over folds)*

| vcd | tcd | substrate | byproduct | **average of 4** |
|---|---|---|---|---|
| 8.6 % | 8.7 % | 12.0 % | 12.9 % | **10.5 % (+/- 0.9)** |

- On average the prediction is about 10 % away from the truth. This is the **baseline**: any model must beat it, otherwise it has learned nothing about the individual batch.
- Almost all of the structure in the data is *time*: VCD goes from about 1 to 31 and back to 22, substrate falls and crashes on day 11, byproduct rises and falls. Knowing the day already explains most of it.
- What is left (10.5 %) is the *batch-to-batch* difference plus measurement noise. That is the room the ML models work in. Ridge, gbm and blend start from this curve and try to explain why one batch is above or below it. Their gain so far is 10.5 -> about 9.3, mostly in VCD/TCD (8.6 -> 7.2).
- The +/- 0.9 means a change smaller than about 1 point cannot be told apart from luck with only 39 batches.

## 4. What the mean curve looks like
Grey = every dev batch. Black = the mean curve (the prediction).

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 7))
for a, c in zip(ax.ravel(), TARGETS):
    for _, g in dev.groupby("batch_id"):
        a.plot(g.elapsed_time, g[c], color="0.8", lw=0.7)
    m = oof.groupby("day")[f"pred_{c}"].mean()
    a.plot(m.index, m.values, "k", lw=2.5)
    a.set_title(c); a.set_xlabel("day")
plt.tight_layout(); plt.show()

## 5. Where is the mean curve wrong? (% error by variable and day)
Out-of-fold errors from the CV above. Large numbers mark the days where the batch differences matter most, and where a better model could gain.

In [ ]:
err = 100 * (oof[[f"pred_{c}" for c in TARGETS]].values - oof[TARGETS].values) / oof[TARGETS].values
err = pd.DataFrame(np.abs(err), columns=TARGETS, index=oof.index); err["day"] = oof.day
by_day = err.groupby("day")[TARGETS].mean().round(1)
by_day

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
im = ax.imshow(by_day.T.values, aspect="auto", cmap="Reds")
ax.set_xticks(range(len(by_day))); ax.set_xticklabels(by_day.index)
ax.set_yticks(range(4)); ax.set_yticklabels(TARGETS); ax.set_xlabel("day")
for i in range(4):
    for j in range(len(by_day)):
        ax.text(j, i, f"{by_day.values[j, i]:.0f}", ha="center", va="center", fontsize=7)
plt.colorbar(im, label="mean abs % error"); plt.tight_layout(); plt.show()

## 6. Interpretation of the error table

Values are the average % error of the mean curve on each day, out of fold.

- **VCD / TCD: about 7-10 % on every day.** Fairly even. The batches differ in how fast they grow, which depends on the day-0 values. That is why the ML models already gain here.
- **Substrate: tiny early (about 3 % on days 1-2), then 10-13 % on days 4-10, and a huge spike on day 11 (34 %).** Early on substrate is simply what was put in, so it is predictable. The day-11 crash happens after `feed_b` is cut on day 10; how deep the dip goes differs a lot between batches, and the values there are small (about 0.9), so a small absolute miss is a large % error.
- **Byproduct: grows steadily from 4 % to 27 % on day 13.** The values get small late in the run (about 0.5), so the same absolute spread means a bigger %. Some single points (e.g. the last-day jumps seen in Batch29 and Batch34) can also inflate it.

**What this tells us**
1. The two cells that hurt most are **substrate day 11 (34 %)** and **byproduct day 13 (27 %)**. Small values dominate the leaderboard score, so these matter most.
2. VCD/TCD is already evenly handled; the next gains are in **substrate and late byproduct**.
3. A good next experiment is timing features for the feeds (feed in the last 1-2 days, days since the last substrate bolus), aimed at the substrate days 4-13.
4. Some of this error is measurement noise that no model can remove. It is worth estimating how much before chasing small gains.